In [13]:
import pandas as pd
import pdfplumber
import numpy as np

In [ ]:
# Loading dataset using pandas
df = pd.read_csv("../data/pcg23Public/epcg23.csv", low_memory=False)
print(df.shape)
df.head()

(94606, 548)


,ACADV,ACCAR,ACCCEP,ACCHG,ACDRG,ACEM,ACFPT,ACGRD,ACINT,ACLIC,...,WRKGP,WTRCHOC,WTREASN,WTREM,WTRLIC,WTROPPS,WTROT,WTRPERS,WTRSKL,WTSURVY
0,L,L,L,L,L,L,L,L,L,L,...,Y,L,L,L,L,L,L,L,L,5466.1729
1,Y,Y,N,Y,0,N,3,Y,N,N,...,Y,Y,1,Y,N,Y,N,N,Y,46.4920
2,L,L,L,L,L,L,L,L,L,L,...,Y,N,5,Y,Y,Y,N,Y,Y,4520.0847
3,L,L,L,L,L,L,L,L,L,L,...,Y,N,1,Y,N,N,N,N,Y,9443.6334
4,L,L,L,L,L,L,L,L,L,L,...,N,L,L,L,L,L,L,L,L,7200.3925


In [7]:

targets = ["N2OCPRMG", "N3OCPR", "N3OCPRX", "Taxonomy of Occupations", "JOBSATIS"]

with pdfplumber.open("../data/pcg23Public/Ppcg23.pdf") as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        for t in targets:
            if t in text:
                print(f"--- page {i+1}: found '{t}' ---")

--- page 8: found 'JOBSATIS' ---
--- page 10: found 'N2OCPRMG' ---
--- page 10: found 'N3OCPR' ---
--- page 10: found 'N3OCPRX' ---
--- page 179: found 'JOBSATIS' ---
--- page 291: found 'N2OCPRMG' ---
--- page 300: found 'N3OCPR' ---
--- page 303: found 'N3OCPR' ---
--- page 304: found 'N3OCPR' ---
--- page 304: found 'N3OCPRX' ---


# Cleaning the Dataset

In [9]:
# Subsetting to only the columns we want to keep for modeling
keep_cols = [
    # id / weight
    "OBSNUM", "WTSURVY",
    # target
    "JOBSATIS",
    # satisfaction facets — keep for validation, NOT as baseline predictors (leakage risk)
    "SATADV", "SATBEN", "SATCHAL", "SATIND", "SATLOC", "SATRESP", "SATSAL", "SATSEC", "SATSOC",
    # occupation / fit
    "N3OCPR", "OCEDRLP", "MGRNAT", "MGROTH", "MGRSOC",
    # employer / job characteristics
    "NEDTP", "INDCODE", "JOBINS", "JOBPENS", "JOBPROFT", "JOBVAC",
    # degree field
    "NDGMEMG",
    # baseline: demographics
    "AGE", "SEX_2023", "RACETHM", "HISPANIC", "CTZN", "MARSTA",
    # baseline: work
    "SALARY", "EARN", "HRSWK", "WKSWK", "WKSYR", "LFSTAT", "TELEC", "TELEFR", "EMSIZE",
]

present = [c for c in keep_cols if c in df.columns]
absent = [c for c in keep_cols if c not in df.columns]
print(f"{len(present)}/{len(keep_cols)} found.")
if absent:
    print("Not in file — double check exact names against your header/codebook:", absent)

work = df[present].copy()

39/39 found.


In [10]:
# Inspecting columns for missing values and data types
for col in present:
    print(f"--- {col} ({work[col].dtype}) ---")
    print(work[col].value_counts(dropna=False).head(8))
    print()

--- OBSNUM (int64) ---
OBSNUM
2023054942    1
2023084844    1
2023029424    1
2023022385    1
2023042159    1
2023092769    1
2023054037    1
2023077355    1
Name: count, dtype: int64

--- WTSURVY (float64) ---
WTSURVY
4204.3280     22
5691.2597     14
25099.6752    13
7711.0840     12
6190.6405     12
9430.1726     11
6397.8294     11
5111.6641     10
Name: count, dtype: int64

--- JOBSATIS (object) ---
JOBSATIS
2    36861
1    35058
L    14806
3     6305
4     1576
Name: count, dtype: int64

--- SATADV (object) ---
SATADV
2    36062
1    21052
3    16341
L    14806
4     6345
Name: count, dtype: int64

--- SATBEN (object) ---
SATBEN
2    34353
1    31071
L    14806
3     9342
4     5034
Name: count, dtype: int64

--- SATCHAL (object) ---
SATCHAL
1    36649
2    31789
L    14806
3     8475
4     2887
Name: count, dtype: int64

--- SATIND (object) ---
SATIND
1    49783
2    24101
L    14806
3     4273
4     1643
Name: count, dtype: int64

--- SATLOC (object) ---
SATLOC
1    48382
2    

In [12]:
# Filtering to only those who are employed (LFSTAT == 1)
employed = cleaned = work[work["LFSTAT"] == 1].copy()
print(f"{len(employed)} of {len(work)} rows are employed (LFSTAT==1)")

# sanity check: this should eliminate (almost) all the L / sentinel rows we saw
for col in ["JOBSATIS", "SATADV", "N3OCPR", "SALARY", "HRSWK"]:
    print(col, employed[col].isin(["L", 999989, 9999998, 98]).sum())

79800 of 94606 rows are employed (LFSTAT==1)
JOBSATIS 0
SATADV 0
N3OCPR 0
SALARY 0
HRSWK 0


In [14]:
# Applying confirmed sentinel mappings to the employed subset
sentinel_map = {
    # letter-coded logical skip
    "JOBSATIS": ["L"], "SATADV": ["L"], "SATBEN": ["L"], "SATCHAL": ["L"], "SATIND": ["L"],
    "SATLOC": ["L"], "SATRESP": ["L"], "SATSAL": ["L"], "SATSEC": ["L"], "SATSOC": ["L"],
    "OCEDRLP": ["L"], "MGRNAT": ["L"], "MGROTH": ["L"], "MGRSOC": ["L"], "NEDTP": ["L"],
    "JOBINS": ["L"], "JOBPENS": ["L"], "JOBPROFT": ["L"], "JOBVAC": ["L"],
    "WKSYR": ["L"], "TELEC": ["L"], "TELEFR": ["L"], "EMSIZE": ["L"],
    # numeric sentinels (confirmed from your value_counts)
    "N3OCPR": [999989],
    "INDCODE": [9998],
    "SALARY": [9999998],
    "EARN": [9999998],
    "HRSWK": [98],
    "WKSWK": [98],
}

def apply_sentinels(frame, mapping):
    frame = frame.copy()
    hits = {}
    for col, bad_vals in mapping.items():
        if col not in frame.columns:
            continue
        mask = frame[col].isin(bad_vals)
        hits[col] = int(mask.sum())
        frame.loc[mask, col] = np.nan
    return frame, hits

employed_clean, hits = apply_sentinels(employed, sentinel_map)
pd.Series(hits).sort_values(ascending=False)

TELEFR      35474
EARN           99
JOBSATIS        0
JOBINS          0
HRSWK           0
SALARY          0
INDCODE         0
N3OCPR          0
EMSIZE          0
TELEC           0
WKSYR           0
JOBVAC          0
JOBPROFT        0
JOBPENS         0
NEDTP           0
SATADV          0
MGRSOC          0
MGROTH          0
MGRNAT          0
OCEDRLP         0
SATSOC          0
SATSEC          0
SATSAL          0
SATRESP         0
SATLOC          0
SATIND          0
SATCHAL         0
SATBEN          0
WKSWK           0
dtype: int64

In [15]:
# dtype conversions
ordinal_like = ["JOBSATIS", "SATADV", "SATBEN", "SATCHAL", "SATIND", "SATLOC",
                "SATRESP", "SATSAL", "SATSEC", "SATSOC", "OCEDRLP", "NEDTP",
                "TELEC", "TELEFR", "EMSIZE", "WKSYR"]

for col in ordinal_like:
    employed_clean[col] = pd.to_numeric(employed_clean[col], errors="coerce")

employed_clean[ordinal_like].dtypes

/var/folders/ny/p5hb2xbx46g25_c_0lg5mhyc0000gn/T/ipykernel_51065/791195553.py:7: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  employed_clean[col] = pd.to_numeric(employed_clean[col], errors="coerce")


JOBSATIS      int64
SATADV        int64
SATBEN        int64
SATCHAL       int64
SATIND        int64
SATLOC        int64
SATRESP       int64
SATSAL        int64
SATSEC        int64
SATSOC        int64
OCEDRLP       int64
NEDTP         int64
TELEC         int64
TELEFR      float64
EMSIZE        int64
WKSYR         int64
dtype: object